<a href="https://colab.research.google.com/github/blacklack547-hash/game-playtime-dashboard/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd

# Load the zipped CSV straight from your GitHub repository raw link
url = "https://raw.githubusercontent.com/blacklack547-hash/game-playtime-dashboard/main/hltb_dataset_normalized.zip"

# Pandas automatically detects the .zip compression and extracts it in memory!
df = pd.read_csv(url)

In [8]:
!pip install streamlit pandas plotly

In [9]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
changed 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [10]:
%%writefile etl_pipeline.py
import pandas as pd
import sqlite3
import os

print("=== STARTING HLTB PLAYTIME ETL PIPELINE ===")
csv_filename = 'hltb_dataset_normalized.csv'

# Verification check
if not os.path.exists(csv_filename):
    raise FileNotFoundError(f"Missing '{csv_filename}'! Please upload it to your Colab sidebar and ensure it is named exactly '{csv_filename}'.")

# 1. Extract raw data
df = pd.read_csv(csv_filename)
print(f"✔ Successfully extracted {len(df)} rows from raw CSV.")

# 2. Curate & Transform Columns (Standardizing layout names to lowercase)
df.columns = df.columns.str.replace(' ', '_').str.lower().str.strip()

# Safely convert time metrics to numeric format and fill blanks with column medians
playtime_cols = ['main_story', 'main_plus_sides', 'completionist', 'all_styles']
for col in playtime_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

# Standardize string fields to lowercase to avoid search mismatch errors
text_cols = ['name', 'type', 'platform', 'genres', 'release_date', 'source_url']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().str.strip()

# 3. Load into an optimized local relational database
conn = sqlite3.connect('backlog.db')
df.to_sql('playtimes', conn, if_exists='replace', index=False)
conn.close()

print("=== ETL PIPELINE PROCESS COMPLETED SUCCESSFULLY ===")

Overwriting etl_pipeline.py


In [11]:
!python etl_pipeline.py

=== STARTING HLTB PLAYTIME ETL PIPELINE ===
✔ Successfully extracted 166754 rows from raw CSV.
=== ETL PIPELINE PROCESS COMPLETED SUCCESSFULLY ===


In [12]:
%%writefile app.py
import os
import zipfile

# Unzip database automatically on Streamlit Cloud startup
if not os.path.exists('backlog.db') and os.path.exists('backlog.zip'):
    with zipfile.ZipFile('backlog.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
import streamlit as st
import sqlite3
import pandas as pd

# Set up dashboard visual workspace with a clean layout
st.set_page_config(page_title="Game Playtime Analytics Dashboard", page_icon="🎮", layout="wide")

# Pull data directly from local relational database tables
@st.cache_data
def load_optimized_data():
    try:
        conn = sqlite3.connect('backlog.db')
        # 💡 FIX: Only pull games that actually have recorded playtime data
        data = pd.read_sql_query("SELECT * FROM playtimes WHERE main_story > 0", conn)
        conn.close()
        return data
    except Exception:
        return pd.DataFrame()

df = load_optimized_data()

if not df.empty:
    st.title("🎮 Video Game Playtime Curation & Analytics Platform")
    st.markdown("Query playtime playstyles, check release records, and browse game information seamlessly.")
    st.markdown("---")

    # High Level Summary Cards (Now filtering out 0-hour placeholder values!)
    kpi1, kpi2, kpi3 = st.columns(3)
    with kpi1:
        st.markdown("### 📊 Total Cataloged Titles")
        st.markdown(f"## **{len(df):,}**")
    with kpi2:
        st.markdown("### ⏱️ Average Main Story")
        # 💡 FIX: Only calculate mean for rows where main_story is greater than 0
        true_main_story = df[df['main_story'] > 0]['main_story'].mean()
        st.markdown(f"## **{true_main_story:.1f} Hours**")
    with kpi3:
        st.markdown("### 🏆 Average 100% Run")
        # 💡 FIX: Only calculate mean for rows where completionist is greater than 0
        true_completionist = df[df['completionist'] > 0]['completionist'].mean()
        st.markdown(f"## **{true_completionist:.1f} Hours**")

    st.markdown("---")

    # Analytics Leaderboard Deck (Pure text insights)
    st.subheader("📈 Playtime Records & Data Insights")
    st.markdown("Key structural milestones extracted directly from your database table:")

    c_longest, c_shortest = st.columns(2)
    with c_longest:
        st.markdown("#### 🏆 Top 5 Longest Games (100% Completion)")
        longest_games = df[df['completionist'] > 0].sort_values(by='completionist', ascending=False).head(5)
        for idx, row in longest_games.iterrows():
            st.markdown(f"• **{str(row['name']).title()}** ({str(row['type']).upper()}) — **{row['completionist']} hrs**")

    with c_shortest:
        st.markdown("#### ⚡ Top 5 Quickest Games (Main Story)")
        shortest_games = df[df['main_story'] > 0].sort_values(by='main_story', ascending=True).head(5)
        for idx, row in shortest_games.iterrows():
            st.markdown(f"• **{str(row['name']).title()}** ({str(row['type']).upper()}) — **{row['main_story']} hrs**")

    st.markdown("---")

    # Browse Catalog Layout (Reliable pure text deck)
    st.subheader("🗂️ Browse Your Cataloged Game Profiles")
    st.markdown("Snapshot overview of games extracted directly from your optimized local database storage:")

    preview_df = df.head(40)
    for idx, row in preview_df.iterrows():
        st.markdown(f"### 🕹️ {str(row['name']).title()}")

        c1, c2, c3 = st.columns(3)
        with c1:
            st.markdown(f"• **Main Story:** {row['main_story']} hrs")
            st.markdown(f"• **Main + Sides:** {row['main_plus_sides']} hrs")
        with c2:
            st.markdown(f"• **100% Run:** {row['completionist']} hrs")
            st.markdown(f"• **Average Style:** {row['all_styles']} hrs")
        with c3:
            st.markdown(f"• **Release Date:** {str(row['release_date']).title()}")
            st.markdown(f"• **Category:** {str(row['type']).upper()}")
        st.markdown(" ")

    st.markdown("---")

    # Dataset Spreadsheet View -> Completely static HTML table element
    st.subheader("📋 Curated Dataset Snapshot Table")
    st.table(df[['name', 'type', 'platform', 'main_story', 'completionist', 'release_date']].head(25))

else:
    st.error("⚠️ The SQLite database is empty. Please execute your 'etl_pipeline.py' script cell before launching the app.")

Overwriting app.py


In [13]:
#kill all stuck background tunnels
!pkill -f streamlit
!pkill -f lt
!pkill -f ssh
!pkill -f cloudflared
!sleep 2

#Download the official Cloudflare tunnel engine
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

#Launch Streamlit and create Cloudflare bridge
!streamlit run app.py --server.port 8501 --server.headless true & ./cloudflared tunnel --url http://localhost:8501

2026-07-27T12:37:52Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-27T12:37:52Z INF Requesting new quick Tunnel on trycloudflare.com...


2026-07-27T12:37:55Z INF +--------------------------------------------------------------------------------------------+
2026-07-27T12:37:55Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-27T12:37:55Z INF |  https://walt-tracked-treasury-ciao.trycloudflare.co